## Завдання 1
Звантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset

In [2]:
import pandas as pd
import numpy as np

def load_power_data(filepath="household_power_consumption.txt"):
    df = pd.read_csv(filepath, sep=';', na_values=['?'], low_memory=False)
    
    return df

df_power_raw = load_power_data()
print("Дані завантажено! Розмір сирого датасету:", df_power_raw.shape)
display(df_power_raw.head())

Дані завантажено! Розмір сирого датасету: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Завдання 2
Здійснити data cleaning

In [3]:
def clean_power_data(df):

    df_cleaned = df.copy()
    df_cleaned = df_cleaned.dropna()
    df_cleaned = df_cleaned.reset_index(drop=True)
    
    return df_cleaned
    
df_power = clean_power_data(df_power_raw)
print("Дані очищено! Новий розмір датасету:", df_power.shape)
display(df_power.head())

Дані очищено! Новий розмір датасету: (2049280, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Завдання 3
Окремими функціями сформувати вибірки

**3.1** Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.


In [4]:
def filter_high_active_power(df):
    result = df[df['Global_active_power'] > 5.0]
    return result

print("Профілювання часу виконання:")
%timeit filter_high_active_power(df_power)

high_power_df = filter_high_active_power(df_power)
print(f"\nЗнайдено записів: {len(high_power_df)}")
display(high_power_df.head())

Профілювання часу виконання:
6.95 ms ± 312 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

Знайдено записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


**3.2** Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер

In [5]:
def filter_intensity_and_appliances(df):
    condition = (df['Global_intensity'] >= 19.0) & \
                (df['Global_intensity'] <= 20.0) & \
                (df['Sub_metering_2'] > df['Sub_metering_3'])
    
    return df[condition]

print("Профілювання часу виконання:")
%timeit filter_intensity_and_appliances(df_power)

appliances_df = filter_intensity_and_appliances(df_power)
print(f"\nЗнайдено записів: {len(appliances_df)}")
display(appliances_df.head())

Профілювання часу виконання:
18.3 ms ± 1.93 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


**3.3** Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [6]:
def sample_and_calculate_means(df):
    sampled_df = df.sample(n=500000, replace=False, random_state=42)
    
    means = {
        'Середнє Sub_metering_1': [sampled_df['Sub_metering_1'].mean()],
        'Середнє Sub_metering_2': [sampled_df['Sub_metering_2'].mean()],
        'Середнє Sub_metering_3': [sampled_df['Sub_metering_3'].mean()]
    }
    
    return pd.DataFrame(means)

print("Профілювання часу виконання:")
%timeit sample_and_calculate_means(df_power)

means_df = sample_and_calculate_means(df_power)
print("\nСередні величини споживання для 500 000 випадкових записів:")
display(means_df)

Профілювання часу виконання:
250 ms ± 27.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Середні величини споживання для 500 000 випадкових записів:


,Середнє Sub_metering_1,Середнє Sub_metering_2,Середнє Sub_metering_3
0,1.119258,1.308912,6.45295


**3.4** Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [7]:
def complex_evening_filter(df):
    cond_time = df['Time'] >= '18:00:00'
    cond_power = df['Global_active_power'] > 6.0
    
    cond_group2_max = (df['Sub_metering_2'] > df['Sub_metering_1']) & \
                      (df['Sub_metering_2'] > df['Sub_metering_3'])
                      
    filtered_df = df[cond_time & cond_power & cond_group2_max]
    
    if filtered_df.empty:
        return filtered_df
        
    mid_index = len(filtered_df) // 2
    first_half = filtered_df.iloc[:mid_index]
    second_half = filtered_df.iloc[mid_index:]
    
    result_first = first_half.iloc[::3]
    result_second = second_half.iloc[::4]
    
    final_result = pd.concat([result_first, result_second])
    
    return final_result

print("Профілювання часу виконання:")
%timeit complex_evening_filter(df_power)

evening_df = complex_evening_filter(df_power)
print(f"\nЗнайдено записів після всіх маніпуляцій: {len(evening_df)}")
display(evening_df.head(10))

Профілювання часу виконання:
270 ms ± 18.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Знайдено записів після всіх маніпуляцій: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17492,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17496,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17499,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0
17502,28/12/2006,21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0
17505,28/12/2006,21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0
17508,28/12/2006,21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0
17511,28/12/2006,21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0
18950,29/12/2006,21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0


## Завдання 4
Пронормувати та стандартизувати вибраний датасет

In [8]:
def normalize_and_standardize(df):
    numeric_df = df.select_dtypes(include=['float64', 'int64'])
    
    normalized_df = (numeric_df - numeric_df.min()) / (numeric_df.max() - numeric_df.min())
    
    standardized_df = (numeric_df - numeric_df.mean()) / numeric_df.std()
    
    return normalized_df, standardized_df

print("Профілювання часу виконання:")
%timeit normalize_and_standardize(df_power)

norm_df, std_df = normalize_and_standardize(df_power)

print("\nНормовані дані (Min-Max) - перші 5 записів:")
display(norm_df.head())

print("\nСтандартизовані дані (Z-score) - перші 5 записів:")
display(std_df.head())

Профілювання часу виконання:
914 ms ± 40.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Нормовані дані (Min-Max) - перші 5 записів:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387



Стандартизовані дані (Z-score) - перші 5 записів:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955076,2.610720,-1.851816,3.098788,-0.182337,-0.051274,1.249420
1,4.037084,2.770405,-2.225274,4.133799,-0.182337,-0.051274,1.130897
2,4.050325,3.320431,-2.330213,4.133799,-0.182337,0.120487,1.249420
3,4.063566,3.355916,-2.191323,4.133799,-0.182337,-0.051274,1.249420
4,2.434881,3.586572,-1.592555,2.513781,-0.182337,-0.051274,1.249420


## Завдання 5
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [9]:
def calculate_correlations(df, col1, col2):
    pearson_corr = df[col1].corr(df[col2], method='pearson')
    
    spearman_corr = df[col1].corr(df[col2], method='spearman')
    
    result_df = pd.DataFrame({
        'Атрибути': [f"{col1} та {col2}", f"{col1} та {col2}"],
        'Тип кореляції': ['Пірсона', 'Спірмена'],
        'Значення': [pearson_corr, spearman_corr]
    })
    
    return result_df

print("Профілювання часу виконання:")
%timeit calculate_correlations(df_power, 'Global_active_power', 'Global_intensity')

correlations_df = calculate_correlations(df_power, 'Global_active_power', 'Global_intensity')
print("\nКоефіцієнти кореляції:")
display(correlations_df)

Профілювання часу виконання:
764 ms ± 17.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Коефіцієнти кореляції:


,Атрибути,Тип кореляції,Значення
0,Global_active_power та Global_intensity,Пірсона,0.998889
1,Global_active_power та Global_intensity,Спірмена,0.995372


## Завдання 6
Провести One Hot Encoding категоріального атрибута.

In [10]:
def apply_one_hot_encoding(df):
    df_subset = df[['Date', 'Time', 'Global_active_power']].copy()
    
    df_subset['Month'] = df_subset['Date'].str.split('/').str[1]
    
    encoded_df = pd.get_dummies(df_subset, columns=['Month'], prefix='Month', dtype=int)
    
    return encoded_df

print("Профілювання часу виконання:")
%timeit apply_one_hot_encoding(df_power)

ohe_df = apply_one_hot_encoding(df_power)
print("\nРезультат One Hot Encoding (перші 5 записів):")
display(ohe_df.head())

Профілювання часу виконання:
2.65 s ± 129 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Результат One Hot Encoding (перші 5 записів):


,Date,Time,Global_active_power,Month_1,Month_10,Month_11,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
0,16/12/2006,17:24:00,4.216,0,0,0,1,0,0,0,0,0,0,0,0
1,16/12/2006,17:25:00,5.360,0,0,0,1,0,0,0,0,0,0,0,0
2,16/12/2006,17:26:00,5.374,0,0,0,1,0,0,0,0,0,0,0,0
3,16/12/2006,17:27:00,5.388,0,0,0,1,0,0,0,0,0,0,0,0
4,16/12/2006,17:28:00,3.666,0,0,0,1,0,0,0,0,0,0,0,0
